### BASELINE EXPERIMENTATION
This notebook explores training of the model without any hyperparameter tuning
Logistic Regression is going to be used since it is a binary classification
Random Forest Classifier is going to be used also
For the remaining classifiers we use Lazy Predict to train multiple classifiers at a time and ranks them based on their accuracy
we going to use mlflow for tracking the experimentation 
Since there is a class imbalnace we use apply an oversampling technique on the minority class

In [1]:
#import dependencies
import pandas as pd
import numpy as np
import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from resuable_code_block import main
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lazypredict.Supervised import LazyClassifier
from sklearn.metrics import accuracy_score,classification_report
import os


In [2]:
#reading the data
data = pd.read_csv(r'Cleaned_data.csv')
data.head(5)

,label,text_preprocessed
0,human,talk gene editing ethic sec feel like people c...
1,human,update election integrity concern complicated ...
2,ai,analyst closely watching development related c...
3,human,paper examines genomic research breakthrough c...
4,ai,existing literature student debt crisis predom...


In [3]:
# labelling encoding

encoder = LabelEncoder()
data['label'] = encoder.fit_transform(data['label'])

In [4]:
data['label'].value_counts() 

label
1    1334
0     666
Name: count, dtype: int64

In [5]:
#splitting the data
X = data.drop('label',axis=1)
y = data['label']


In [6]:
# shape of the data
print(f'Shape of the feature dataset {X.shape}')
print(f'Shape of the target dataset {y.shape}')

# print the first text in the feature datasets
print(data['text_preprocessed'][0])
X.head()

Shape of the feature dataset (2000, 1)
Shape of the target dataset (2000,)
talk gene editing ethic sec feel like people completely missing point bigger realize


,text_preprocessed
0,talk gene editing ethic sec feel like people c...
1,update election integrity concern complicated ...
2,analyst closely watching development related c...
3,paper examines genomic research breakthrough c...
4,existing literature student debt crisis predom...


In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f'Shape of X_train {X_train.shape}')
print(f'shpae of Y_train {y_train.shape}')
print(f'shape of X_test {X_test.shape}')
print(f'Shape of X_test {X_test.shape}')


Shape of X_train (1600, 1)
shpae of Y_train (1600,)
shape of X_test (400, 1)
Shape of X_test (400, 1)


In [8]:
X_train_res,y_train_res, X_test_vec = main(X_train,X_test,y_train)
print(f'Shape of X_train_res {X_train_res.shape}')
print(f'Shape of X_train_res {y_train_res.shape}')
print(f'Shape of X_train_res {X_test_vec.shape}')


Shape of X_train_res (2134, 500)
Shape of X_train_res (2134,)
Shape of X_train_res (400, 500)


In [20]:
mlflow.set_experiment('Baseline_Experimentations for different models')

2026/05/13 08:07:26 INFO mlflow.tracking.fluent: Experiment with name 'Baseline_Experimentations for different models' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/367010398081178386', creation_time=1778684846649, experiment_id='367010398081178386', last_update_time=1778684846649, lifecycle_stage='active', name='Baseline_Experimentations for different models', tags={}>

In [21]:
#set tracking experiment for mlflow
mlflow.set_tracking_uri('http://127.0.0.1:5000')

## LOGISTIC REGRESSION

In [26]:
def log(X_train,X_test,y_train,y_test):
    try:
        with mlflow.start_run(run_name='Baseline Experimentation with Logistic regression'):
            tfidf_params = {
                'tfidf_ngram_range': (1, 2),
                'tfidf_max_features': 500,
                'tfidf_max_df': 0.9
            }
            
            smote_params = {
                'smote_random_state': 42,
                'smote_k_neighbors': 7
            }
            
            mlflow.log_params(tfidf_params)
            mlflow.log_params(smote_params)
            
            
            X_train_res,y_train_res, X_test_vec = main(X_train,X_test,y_train)

            model = LogisticRegression()

            #training the model
            model.fit(X_train_res,y_train_res)

            #testin the model
            y_pred = model.predict(X_test_vec)

            #accuracy of the model
            accuracy = accuracy_score(y_test,y_pred)
            report = classification_report(y_test,y_pred,output_dict=True)

            #logging the report parameters
            for label,metrics in report.items():
                if isinstance(metrics,dict):
                    for metric,value in metrics.items():
                        mlflow.log_metric(f'test_{label}_{metric}',value)
            
            mlflow.sklearn.log_model(model,'Logistic Regression')

            print(f'Accuracy of the Logistic Regression Model {accuracy}')
    except Exception as e:
        print('An error occured',e)



In [27]:
log(X_train,X_test,y_train,y_test)

2026/05/13 08:10:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 08:10:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Accuracy of the Logistic Regression Model 1.0
🏃 View run Baseline Experimentation with Logistic regression at: http://127.0.0.1:5000/#/experiments/367010398081178386/runs/10174eb3b3dd40b4b0d315263e07a5b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/367010398081178386


## RANDOM FOREST CLASSIFIER

In [28]:
def random_forest(X_train,X_test,y_train,y_test):
    try:
        with mlflow.start_run(run_name='Baseline Experimentation with Random Forest'):
            tfidf_params = {
                'tfidf_ngram_range': (1, 2),
                'tfidf_max_features': 500,
                'tfidf_max_df': 0.9
            }
            
            smote_params = {
                'smote_random_state': 42,
                'smote_k_neighbors': 7
            }
            
            mlflow.log_params(tfidf_params)
            mlflow.log_params(smote_params)
            
            
            X_train_res,y_train_res, X_test_vec = main(X_train,X_test,y_train)

            model = RandomForestClassifier()

            #training the model
            model.fit(X_train_res,y_train_res)

            #testin the model
            y_pred = model.predict(X_test_vec)

            #accuracy of the model
            accuracy = accuracy_score(y_test,y_pred)
            report = classification_report(y_test,y_pred,output_dict=True)

            #logging the report parameters
            for label,metrics in report.items():
                if isinstance(metrics,dict):
                    for metric,value in metrics.items():
                        mlflow.log_metric(f'test_{label}_{metric}',value)
            
            mlflow.sklearn.log_model(model, 'Random Forest Classifier')

            print(f'Accuracy of the Random Forest {accuracy}')
    except Exception as e:
        print('An error occured',e)



In [29]:
random_forest(X_train,X_test,y_train,y_test)

2026/05/13 08:11:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 08:12:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Accuracy of the Random Forest 1.0
🏃 View run Baseline Experimentation with Random Forest at: http://127.0.0.1:5000/#/experiments/367010398081178386/runs/74b9770093bd4261b6232cab513ab3ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/367010398081178386


## LAZY PREDICT CLASSIFIER

In [ ]:
def lazy_predict(X_train, X_test, y_train, y_test):
    try:
        
        mlflow.autolog(disable=True)
        
        # End any stale runs
        if mlflow.active_run():
            mlflow.end_run()

        with mlflow.start_run(run_name='Baseline Experimentation with Lazy_predict'):
            tfidf_params = {
                'tfidf_ngram_range': (1, 2),
                'tfidf_max_features': 500,
                'tfidf_max_df': 0.9
            }
            
            smote_params = {
                'smote_random_state': 42,
                'smote_k_neighbors': 7
            }
            
            mlflow.log_params(tfidf_params)
            mlflow.log_params(smote_params)
            
            X_train_res, y_train_res, X_test_vec = main(X_train, X_test, y_train)

            model = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
            
            # End the run BEFORE LazyClassifier fits
            # so no active run can conflict
            mlflow.end_run()
            
            models, predictions = model.fit(
                X_train_res.toarray(), X_test_vec.toarray(), y_train_res, y_test
            )
            
            print("Models shape:", models.shape)
            print(models.head())
            
            # Now start a NEW run to log the results
            with mlflow.start_run(run_name='Baseline Experimentation with Lazy_predict'):
                mlflow.log_params(tfidf_params)
                mlflow.log_params(smote_params)
                
                report = models.to_dict(orient='index')

                for label, metrics in report.items():
                    if isinstance(metrics, dict):
                        clean_label = label.replace(" ", "_").replace("(", "").replace(")", "")
                        for metric, value in metrics.items():
                            clean_metric = metric.replace(" ", "_")
                            if isinstance(value, (int, float)):
                                mlflow.log_metric(f'test_{clean_label}_{clean_metric}', value)
                
                models.to_csv("summary_metrics.csv")
                mlflow.log_artifact("summary_metrics.csv")
                    
                print(f'Logged metrics for {len(models)} models')
    
    except Exception as e:
        import traceback
        traceback.print_exc()

In [ ]:
lazy_predict(X_train,X_test,y_train,y_test)